In [ ]:
#| hide
from fastgit import *
import os

In [ ]:
#| hide
os.environ.update(GIT_AUTHOR_NAME='fastgit', GIT_AUTHOR_EMAIL='fastgit@example.com',
    GIT_COMMITTER_NAME='fastgit', GIT_COMMITTER_EMAIL='fastgit@example.com')

# fastgit

> Use git from python, fast

#| export
`fastgit` is a thin Python wrapper for the `git` CLI: one `Git` object whose attribute calls run git commands, so if you know git, you already know fastgit. There is no reimplementation of git internals and no object model to learn -- commands return git's own output as (subclassed) `str`s.

It is designed for interactive use and automation alike: errors print tersely by default (like git itself) or raise on request, exit codes that mean "no" rather than "failed" are returned normally, and passing `sync=False` gives an async client for servers and concurrent code.

## Usage

### Installation

Install latest from [pypi][pypi]

```sh
$ pip install fastgit
```

[pypi]: https://pypi.org/project/fastgit/

### How to use

Create a `Git` for any directory; every command runs with that directory as its working tree. Method names map to git subcommands, and results come back as stripped strings:

In [ ]:
import shutil, tempfile

In [ ]:
td = tempfile.mkdtemp()
g = Git(td)
g.init(b='main')

'Initialized empty Git repository in /private/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/tmp8zyj76q7/.git/'

In [ ]:
(g.d/'.gitignore').mk_write('*.bak')
g.add('.gitignore')
g.commit(m='add .gitignore')

'[main (root-commit) 5113ce1] add .gitignore\n 1 file changed, 1 insertion(+)\n create mode 100644 .gitignore'

Keyword arguments become options: single-letter names are short options (`n=1` → `-n 1`), longer names long options with underscores turned into dashes, and `True` passes the bare flag:

In [ ]:
g.log(n=1, oneline=True)

'5113ce1 add .gitignore'

You can also pass path arguments after `--` using the `__` parameter:

In [ ]:
g.log('--oneline', __=['.gitignore'])

'5113ce1 add .gitignore'

Frequent queries are properties:

In [ ]:
g.current_branch, g.commits

('main', ['5113ce1 add .gitignore'])

A failed command prints git's message and returns `None`; pass `raise_exc=True` (per call or at init) to raise instead. Where git uses exit 1 to mean "no" rather than "error" -- like `grep` finding nothing -- the output is returned as usual, with the code on `.returncode`:

In [ ]:
res = g.grep('missing')
res.returncode

1

Pass `sync=False` for an async client: the same commands and properties, each returning an awaitable, so a server never blocks its event loop on git:

In [ ]:
ag = Git(td, sync=False)
await ag.last_commit

'add .gitignore'

In [ ]:
#| hide
shutil.rmtree(td)